In [1]:
pip install pandas openpyxl 

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
file_path = r'/Users/valeriia/Desktop/base/sales_validation_practice.xlsx'

df = pd.read_excel(file_path, sheet_name = "Sales Data")

# 1. Обрабатываем пустые значения

missing_values = df[df.isna().any(axis=1)] # isna = пустое ли значение

# 2. Обрабатываем некорректн возраст

invalid_age = df[
    (df["Age"] < 18) |
    (df["Age"] > 100) |
    (df["Age"].isna())
] # | или

# 3. Обрабатываем некорректн email

invalid_email = df[
    df["Email"].isna() | # не должен содержать пробел
    ~df["Email"].astype(str).str.contains("@", na=False) # ~ "или не" И na=False пишем по дефолту
]

# 4. Обрабатываем некорректный телефон
invalid_phone = df[
    df["Phone"].isna() |
    ~df["Phone"].astype(str).str.startswith("996") |
    (df["Phone"].astype(str).str.len() != 12)
]

# 5. Обрабатываем некорректную категорию
allowed_categories = [
    "Electronics",
    "Clothing",
    "Food",
    "Sport"
]

invalid_category = df[
    ~df["Category"].isin(allowed_categories)
]

# 6. Обрабатываем некорректное количество
invalid_quantity = df[
    (df["Quantity"] <= 0) |
    (df["Quantity"].isna())
]

# 7. Обрабатываем некорректную цену
invalid_price = df[
    (df["Price"] <= 0) |
    (df["Price"].isna())
]

# 8. Будущие даты 
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")

future_dates = df[
    df["Order Date"] > pd.Timestamp.today()
]

# 9. Пустые даты 
invalid_dates = df[
    df["Order Date"].isna()
]

# 10. Дубликаты 
duplicates = df[df.duplicated()]

# Сводный отчет 
summary = pd.DataFrame({
    "Validation Check": [
        "Missing Values",
        "Invalid age",
        "Invalid email",
        "Invalid phone",
        "Invalid category",
        "Invalid quantity",
        "Invalid price",
        "Future dates",
        "Invalid dates",
        "Duplicates"
    ],
    "Errors Count": [
        len(missing_values),
        len(invalid_age),
        len(invalid_email),
        len(invalid_phone),
        len(invalid_category),
        len(invalid_quantity),
        len(invalid_price),
        len(future_dates),
        len(invalid_dates),
        len(duplicates)
    ]
}) 

# Создаем валидированные данные 

valid_df = df.copy()

valid_df = valid_df.dropna()

valid_df = valid_df[
    (valid_df["Age"] >= 18) &
    (valid_df["Age"] <= 100)
]

valid_df = valid_df[
    valid_df["Email"].astype(str).str.contains("@", na=False)
]

valid_df = valid_df[
    valid_df["Phone"].astype(str).str.startswith("996") &
    (valid_df["Phone"].astype(str).str.len() == 12)
]

valid_df = valid_df[
    valid_df["Category"].isin(allowed_categories)
]

valid_df = valid_df[
    valid_df["Price"] > 0
]

valid_df = valid_df[
    valid_df["Order Date"] <= pd.Timestamp.today()    
]

valid_df = valid_df.drop_duplicates() 

with pd.ExcelWriter("validation_report.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Original Data", index=False)
    valid_df.to_excel(writer, sheet_name="Validated Data", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)

    missing_values.to_excel(writer, sheet_name="Missing Values", index=False)
    invalid_age.to_excel(writer, sheet_name="Invalid Age", index=False)
    invalid_email.to_excel(writer, sheet_name="Invalid Email", index=False)
    invalid_phone.to_excel(writer, sheet_name="Invalid Phone", index=False)
    invalid_category.to_excel(writer, sheet_name="Invalid Category", index=False)
    invalid_quantity.to_excel(writer, sheet_name="Invalid Quantity", index=False)
    invalid_price.to_excel(writer, sheet_name="Invalid Price", index=False)
    future_dates.to_excel(writer, sheet_name="Future Dates", index=False)
    invalid_dates.to_excel(writer, sheet_name="Invalid Dates", index=False)
    duplicates.to_excel(writer, sheet_name="Duplicates", index=False)

print("Готово")

Готово
